# Hybrid ISP: training on Colab GPU

Use **Runtime > Run all**. The notebook asks for a T4 GPU runtime.
It writes the project code, downloads the data, trains the three models, exports the INT8 model, runs the evaluation and finally downloads `hybrid_isp_colab_results.zip` (checkpoints + results).

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
%%writefile raw_sim.py
"""RAW sensor simulation and the rendering ops shared by all pipelines.

sRGB images are "unprocessed" into camera RAW following Brooks et al.,
"Unprocessing Images for Learned Raw Denoising" (CVPR 2019):
    sRGB -> inverse tone curve -> inverse gamma -> sRGB->camera CCM
         -> inverse white balance -> RGGB mosaic -> exposure (illumination)
         -> shot/read noise -> 10-bit quantisation with black level.
Every step is invertible, so the original sRGB image is the ground truth.
"""
import numpy as np
import torch
import torch.nn.functional as F

# Sensor colour response: XYZ->camera matrix of a real camera (from the
# unprocessing paper) times sRGB(D65)->XYZ, rows normalised so white stays white.
XYZ2CAM = torch.tensor([[1.0234, -0.2969, -0.2266],
                        [-0.5625, 1.6328, -0.0469],
                        [-0.0703, 0.2188, 0.6406]])
RGB2XYZ = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                        [0.2126729, 0.7151522, 0.0721750],
                        [0.0193339, 0.1191920, 0.9503041]])
RGB2CAM = XYZ2CAM @ RGB2XYZ
RGB2CAM = RGB2CAM / RGB2CAM.sum(1, keepdim=True)
CAM2RGB = torch.linalg.inv(RGB2CAM)  # the camera's colour correction matrix

BLACK, WHITE = 64, 1023  # 10-bit sensor
# White balance gains (R, B) of the illuminants the sensor is calibrated for.
# Training samples illuminants from this range; AWB clamps its estimate to it.
WB_RANGE = ((1.6, 2.6), (1.3, 2.1))

# Test-time capture conditions. gain = digital gain applied to reach normal
# brightness (1/exposure), shot = sensor shot-noise level. Training covers
# gain 1..32, so "extreme" is outside the training range on purpose.
CONDITIONS = {
    'daylight': dict(gain=1, shot=2e-4),
    'indoor': dict(gain=4, shot=5e-4),
    'low_light': dict(gain=16, shot=5e-4),
    'night': dict(gain=32, shot=1e-3),
    'extreme': dict(gain=64, shot=1e-3),
}


def read_noise(shot):
    # read noise follows shot noise roughly log-linearly (fit from Brooks et al.)
    return np.exp(2.18 * np.log(shot) + 1.2)


def srgb_to_linear(x):
    x = x.clamp(0, 1)
    x = 0.5 - torch.sin(torch.asin(1 - 2 * x) / 3)  # inverse smoothstep tone curve
    return x.clamp(min=1e-4) ** 2.2


def linear_to_srgb(x):
    """Gamma correction followed by a global S-curve tone map."""
    x = x.clamp(1e-4, 1) ** (1 / 2.2)
    return 3 * x ** 2 - 2 * x ** 3


def apply_ccm(x, m):
    return torch.einsum('ij,njhw->nihw', m.to(x), x)


def pack(x):
    """(N,3,H,W) image -> (N,4,H/2,W/2) RGGB planes [R, Gr, Gb, B]."""
    return F.pixel_unshuffle(x, 2)[:, [0, 5, 6, 11]]


def unpack(x):
    """(N,4,H/2,W/2) RGGB planes -> (N,1,H,W) Bayer mosaic."""
    return F.pixel_shuffle(x, 2)


def unprocess(srgb, wb):
    """sRGB batch -> (white-balanced camera-space linear image, packed RAW before noise).

    wb: (N,3) white balance gains (r, 1, b) of the simulated illuminant.
    """
    cam = apply_ccm(srgb_to_linear(srgb), RGB2CAM)
    inv = (1 / wb)[..., None, None]
    # smoothly keep near-white pixels from losing brightness (as in Brooks et al.)
    mask = ((cam.mean(1, keepdim=True) - 0.9).clamp(min=0) / 0.1) ** 2
    inv = torch.maximum(mask + (1 - mask) * inv, inv)
    return cam, pack(cam * inv)


def add_sensor_noise(raw4, gain, shot, read, generator=None):
    """Under-exposure by 1/gain, heteroscedastic noise and 10-bit quantisation -> DN."""
    x = raw4 / gain
    noise = torch.randn(x.shape, generator=generator, device=x.device)
    x = x + noise * (shot * x.clamp(min=0) + read).sqrt()
    return (x * (WHITE - BLACK) + BLACK).round().clamp(0, WHITE)


def sample_conditions(n):
    """Random illumination / noise / illuminant per training sample."""
    gain = 2 ** (5 * torch.rand(n))  # 1x .. 32x
    log_shot = torch.empty(n).uniform_(np.log(1e-4), np.log(1e-3))
    shot = log_shot.exp()
    read = (2.18 * log_shot + 1.2 + 0.26 * torch.randn(n)).exp()
    wb = torch.stack([torch.empty(n).uniform_(*WB_RANGE[0]), torch.ones(n),
                      torch.empty(n).uniform_(*WB_RANGE[1])], 1)
    return wb, gain, shot, read


def capture(srgb, gain=1.0, shot=2e-4, wb=(2.1, 1.0, 1.7), seed=0):
    """Simulates a phone capture of one sRGB image (H,W,3 float in [0,1]).

    Returns the Bayer mosaic in DN (uint16) and the metadata a real camera
    would store with it (black/white level, as-shot WB, gain, noise profile, CCM).
    """
    read = float(read_noise(shot))
    x = torch.from_numpy(np.ascontiguousarray(srgb)).permute(2, 0, 1)[None].float()
    _, raw4 = unprocess(x, torch.tensor([wb], dtype=torch.float32))
    dn = add_sensor_noise(raw4, gain, shot, read, torch.Generator().manual_seed(seed))
    mosaic = unpack(dn)[0, 0].numpy().astype(np.uint16)
    meta = dict(black=BLACK, white=WHITE, wb=tuple(wb), gain=float(gain),
                noise=(shot, read), cam2rgb=CAM2RGB.numpy(), wb_range=WB_RANGE)
    return mosaic, meta


In [ ]:
%%writefile classical_isp.py
"""Classical ISP stages (NumPy / OpenCV) and the conventional baseline pipeline.

All functions work on float32 images. Bayer data is always RGGB here; real
RAW files with other layouts are shifted to RGGB when loaded.
"""
import cv2
import numpy as np
from scipy.optimize import nnls

RGB2YCC = np.array([[0.299, 0.587, 0.114],
                    [-0.168736, -0.331264, 0.5],
                    [0.5, -0.418688, -0.081312]], np.float32)  # BT.601 full range
YCC2RGB = np.linalg.inv(RGB2YCC).astype(np.float32)


# ---------------------------------------------------------------- RAW preprocessing

def normalize(mosaic, meta):
    """Black level subtraction + white level normalisation (pre-gain, ~[0,1])."""
    return (mosaic.astype(np.float32) - meta['black']) / (meta['white'] - meta['black'])


def planes(m):
    """RGGB mosaic -> R, Gr, Gb, B planes (views)."""
    return m[0::2, 0::2], m[0::2, 1::2], m[1::2, 0::2], m[1::2, 1::2]


def auto_exposure(m, target=0.18):
    """Digital gain that brings mean green to mid grey (used when gain is unknown)."""
    _, gr, gb, _ = planes(m)
    return float(np.clip(target / max((gr.mean() + gb.mean()) / 2, 1e-4), 1, 64))


def gray_world(m, p=1, limits=None):
    """Grey-world AWB on the RAW planes (p > 1: shades of grey, towards white patch).

    limits: ((r_min, r_max), (b_min, b_max)) gains of the illuminants the sensor is
    calibrated for. Estimates outside that range are clamped, as camera AWB does.
    p=1 with clamping had the lowest gain error on DIV2K.
    """
    r, gr, gb, b = planes(m)
    ok = np.maximum(np.maximum(r, gr), np.maximum(gb, b)) < 0.95  # skip clipped pixels
    norm = lambda c: (np.clip(c[ok], 0, None) ** p).mean() ** (1 / p)
    g = (norm(gr) + norm(gb)) / 2
    r, b = g / norm(r), g / norm(b)
    if limits:
        r, b = np.clip(r, *limits[0]), np.clip(b, *limits[1])
    return float(r), 1.0, float(b)


_HP = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32) / 6  # Immerkaer, unit noise gain


def estimate_noise(m, block=8, bins=10):
    """Blind noise profile estimate: fits var(x) = shot * x + read.

    Each Bayer plane is high-pass filtered (cancels smooth signal) and split
    into 8x8 blocks. Per brightness bin, the flattest 5% of blocks give the
    noise variance; texture only adds variance so it is mostly ignored.
    1.77 corrects the low-percentile bias (calibrated on flat test charts).
    """
    xs, vs = [], []
    for p in planes(m):
        p = np.ascontiguousarray(p)
        r = cv2.filter2D(p, -1, _HP)[1:-1, 1:-1]
        h, w = (r.shape[0] // block) * block, (r.shape[1] // block) * block
        blocks = lambda a: a[:h, :w].reshape(h // block, block, w // block, block).mean((1, 3)).ravel()
        vs.append(blocks(r * r))
        xs.append(blocks(p[1:-1, 1:-1]))
    x, v = np.concatenate(xs), np.concatenate(vs)
    ok = (x > -0.02) & (x < 0.9)  # black offset keeps negative noise, so dark blocks are usable
    x, v = x[ok], v[ok]
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    X, V = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (x >= lo) & (x <= hi)
        if s.sum() > 20:
            X.append(max(np.median(x[s]), 0))
            V.append(np.percentile(v[s], 5) * 1.77)
    shot, read = nnls(np.stack([X, np.ones(len(X))], 1), np.array(V))[0]
    return max(float(shot), 1e-6), max(float(read), 1e-9)


def resolve_meta(m, meta):
    """Uses camera metadata when present, otherwise estimates it from the image.

    This is what makes the pipelines adapt to unknown exposure, illuminant and
    sensor noise (e.g. RAW files without a noise profile).
    """
    gain = meta.get('gain') or auto_exposure(m)
    wb = meta.get('wb') or gray_world(m, limits=meta.get('wb_range'))
    noise = meta.get('noise') or estimate_noise(m)
    return gain, wb, noise


def apply_wb(m, wb):
    m = m.copy()
    m[0::2, 0::2] *= wb[0]
    m[1::2, 1::2] *= wb[2]
    return m


# ---------------------------------------------------------------- demosaicing

# Malvar-He-Cutler (2004) gradient-corrected bilinear kernels
_G_AT_RB = np.array([[0, 0, -1, 0, 0],
                     [0, 0, 2, 0, 0],
                     [-1, 2, 4, 2, -1],
                     [0, 0, 2, 0, 0],
                     [0, 0, -1, 0, 0]], np.float32) / 8
_RB_AT_G_ROW = np.array([[0, 0, 0.5, 0, 0],
                         [0, -1, 0, -1, 0],
                         [-1, 4, 5, 4, -1],
                         [0, -1, 0, -1, 0],
                         [0, 0, 0.5, 0, 0]], np.float32) / 8
_RB_AT_G_COL = _RB_AT_G_ROW.T.copy()
_RB_AT_BR = np.array([[0, 0, -1.5, 0, 0],
                      [0, 2, 0, 2, 0],
                      [-1.5, 0, 6, 0, -1.5],
                      [0, 2, 0, 2, 0],
                      [0, 0, -1.5, 0, 0]], np.float32) / 8


def demosaic_mhc(m):
    """RGGB mosaic (H,W) -> RGB (H,W,3) with Malvar-He-Cutler interpolation."""
    # REFLECT_101 keeps the Bayer phase at the borders (row -1 mirrors row 1)
    f = lambda k: cv2.filter2D(m, -1, k, borderType=cv2.BORDER_REFLECT_101)
    g, row, col, diag = f(_G_AT_RB), f(_RB_AT_G_ROW), f(_RB_AT_G_COL), f(_RB_AT_BR)
    out = np.empty(m.shape + (3,), np.float32)
    R, G, B = out[..., 0], out[..., 1], out[..., 2]
    s = (slice(0, None, 2), slice(0, None, 2))   # R sites
    R[s], G[s], B[s] = m[s], g[s], diag[s]
    s = (slice(0, None, 2), slice(1, None, 2))   # G in red rows
    R[s], G[s], B[s] = row[s], m[s], col[s]
    s = (slice(1, None, 2), slice(0, None, 2))   # G in blue rows
    R[s], G[s], B[s] = col[s], m[s], row[s]
    s = (slice(1, None, 2), slice(1, None, 2))   # B sites
    R[s], G[s], B[s] = diag[s], g[s], m[s]
    return out


# ---------------------------------------------------------------- colour

def apply_ccm(rgb, ccm):
    return rgb @ np.asarray(ccm, np.float32).T


def gamma(x):
    return np.clip(x, 1e-4, 1) ** (1 / 2.2)


def tone_map(x):
    """Global S-curve (smoothstep): lifts mid-tone contrast, rolls off highlights."""
    x = np.clip(x, 0, 1)
    return 3 * x ** 2 - 2 * x ** 3


def to_ycc(x):
    return x @ RGB2YCC.T


def from_ycc(x):
    return x @ YCC2RGB.T


# ---------------------------------------------------------------- denoise / sharpen

def display_sigma(gain, wb, noise, ccm, level=0.18):
    """Approximate noise std after gain, WB, CCM and gamma at a mid-grey level."""
    shot, read = noise
    w = np.asarray(wb, np.float32)
    var = shot * gain * w * level + read * (gain * w) ** 2    # per channel after WB
    var = (np.asarray(ccm, np.float32) ** 2) @ var              # CCM mixes the channels
    return float(np.sqrt(var.mean()) * level ** (1 / 2.2 - 1) / 2.2)  # gamma slope


def denoise(x, sigma, k_luma=0.6, k_chroma=4.0):
    """Edge-preserving denoising in YCbCr: non-local means on luma, bilateral on chroma."""
    if sigma < 0.5 / 255:
        return x
    ycc = to_ycc(x)
    y8 = np.clip(ycc[..., 0] * 255, 0, 255).astype(np.uint8)
    ycc[..., 0] = cv2.fastNlMeansDenoising(y8, None, h=k_luma * sigma * 255,
                                           templateWindowSize=7, searchWindowSize=21) / 255.0
    for ch in (1, 2):
        ycc[..., ch] = cv2.bilateralFilter(np.ascontiguousarray(ycc[..., ch]), 9, k_chroma * sigma, 4)
    return from_ycc(ycc)


def sharpen(x, amount=0.5, radius=1.0):
    """Unsharp masking on luma only, so no colour fringes are added."""
    ycc = to_ycc(x)
    y = np.ascontiguousarray(ycc[..., 0])
    ycc[..., 0] = y + amount * (y - cv2.GaussianBlur(y, (0, 0), radius))
    return from_ycc(ycc)


def classical_isp(mosaic, meta, k_luma=0.6, k_chroma=4.0, sharp=0.3):
    """Conventional baseline:
    BLC -> gain -> WB -> demosaic -> CCM -> gamma -> denoise -> tone map -> sharpen.
    Denoise and sharpening strength follow the (known or estimated) noise level.
    """
    m = normalize(mosaic, meta)
    gain, wb, noise = resolve_meta(m, meta)
    m = np.minimum(apply_wb(m * gain, wb), 1)   # saturated highlights -> white
    rgb = np.clip(apply_ccm(demosaic_mhc(m), meta['cam2rgb']), 0, 1)
    x = gamma(rgb)
    sigma = display_sigma(gain, wb, noise, meta['cam2rgb'])
    x = denoise(x, sigma, k_luma, k_chroma)
    x = tone_map(x)
    x = sharpen(x, amount=sharp / (1 + 30 * sigma))  # back off when noisy
    return np.clip(x, 0, 1)


In [ ]:
%%writefile model.py
"""Compact U-Net used for the learned ISP components.

Input : packed RGGB Bayer planes (4) + per-plane noise std map (4), H/2 x W/2
Output: 3-channel image at full resolution H x W (pixel shuffle)

The noise map makes the network adapt to the noise level of each frame
(as in FFDNet / Brooks et al.). The global branch pools the bottleneck over
the whole frame so the network can use scene-level cues (illuminant,
brightness) that a purely local receptive field cannot see.
"""
import torch
import torch.nn as nn


def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU(inplace=True))


class UNet(nn.Module):
    def __init__(self, width=24, cin=8):
        super().__init__()
        c = width
        self.e1, self.e2 = block(cin, c), block(c, 2 * c)
        self.e3, self.e4 = block(2 * c, 4 * c), block(4 * c, 8 * c)
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.glob = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(8 * c, 8 * c, 1), nn.ReLU(inplace=True))
        self.d3, self.d2, self.d1 = block(12 * c, 4 * c), block(6 * c, 2 * c), block(3 * c, c)
        self.head = nn.Conv2d(c, 12, 1)
        self.shuffle = nn.PixelShuffle(2)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        e4 = e4 + self.glob(e4)
        d3 = self.d3(torch.cat([self.up(e4), e3], 1))
        d2 = self.d2(torch.cat([self.up(d3), e2], 1))
        d1 = self.d1(torch.cat([self.up(d2), e1], 1))
        return self.shuffle(self.head(d1))


if __name__ == '__main__':
    for w in (8, 12, 16, 24, 32):
        net = UNet(w)
        out = net(torch.zeros(1, 8, 64, 64))
        assert out.shape == (1, 3, 128, 128)
        print(f'width {w}: {sum(p.numel() for p in net.parameters()) / 1e6:.3f} M params')


In [ ]:
%%writefile hybrid_isp.py
"""Learned and hybrid ISP pipelines.

Hybrid ISP (the proposed design):
    classical front end : BLC, normalisation, exposure gain, white balance,
                          noise profile (metadata or blind estimate)
    learned core        : U-Net does joint demosaicing + denoising on packed
                          Bayer, conditioned on a noise map, as a residual over
                          bilinear demosaicing
    classical back end  : camera CCM, gamma, tone curve, noise-aware sharpening

Learned ISP (end-to-end baseline): the same U-Net maps RAW (+ noise map)
straight to sRGB and has to learn WB, colour correction and tone as well.

Usage:  python hybrid_isp.py photo.dng -o out.png
"""
import argparse
import copy

import numpy as np
import torch
import torch.nn.functional as F

import classical_isp as cl
import raw_sim as rs
from model import UNet


# ---------------------------------------------------------------- shared tensor ops

def net_input(raw4, gain, wb, shot, read):
    """Exposure-normalised (and white-balanced) packed Bayer + per-plane noise std map.

    raw4: (N,4,h,w) normalised RAW planes, gain/shot/read: (N,), wb: (N,3).
    Noise model after gain g and WB gain w:  var = w*g*shot*x + (w*g)^2 * read
    """
    g = gain.view(-1, 1, 1, 1)
    w = wb[:, [0, 1, 1, 2]].view(-1, 4, 1, 1)
    x = (raw4 * g * w).clamp(max=1)
    var = shot.view(-1, 1, 1, 1) * g * w * x.clamp(min=0) + read.view(-1, 1, 1, 1) * (g * w) ** 2
    return torch.cat([x, var.sqrt()], 1)


_K_RB = torch.tensor([[1., 2, 1], [2, 4, 2], [1, 2, 1]]) / 4
_K_G = torch.tensor([[0., 1, 0], [1, 4, 1], [0, 1, 0]]) / 4
_SEL = torch.tensor([[1., 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 1]])  # RGGB plane -> colour


def bilinear_demosaic(raw4):
    """Packed RGGB (N,4,h,w) -> RGB (N,3,2h,2w), normalised convolution (exact at borders)."""
    _, _, h, w = raw4.shape
    mask = F.pixel_shuffle(_SEL.to(raw4).view(1, 12, 1, 1).expand(1, 12, h, w), 2)
    k = torch.stack([_K_RB, _K_G, _K_RB])[:, None].to(raw4)
    sparse = F.pixel_shuffle(raw4, 2) * mask
    return F.conv2d(sparse, k, padding=1, groups=3) / F.conv2d(mask, k, padding=1, groups=3)


def render(cam, ccm=rs.CAM2RGB):
    """Classical back end (differentiable): camera RGB -> CCM -> gamma -> tone curve."""
    return rs.linear_to_srgb(rs.apply_ccm(cam, torch.as_tensor(ccm, dtype=torch.float32)).clamp(0, 1))


# ---------------------------------------------------------------- models

def load_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    net = UNet(ck['width'])
    net.load_state_dict(ck['state'])
    return net.eval()


def quantize_int8(net, calib):
    """Post-training static INT8 quantisation (FX graph mode, oneDNN backend).

    The first conv (raw input range) and the output head (fine residual
    values) are kept in FP32, everything else runs in INT8.
    """
    from torch.ao.quantization import get_default_qconfig_mapping
    from torch.ao.quantization.quantize_fx import convert_fx, prepare_fx
    torch.backends.quantized.engine = 'onednn'
    qmap = get_default_qconfig_mapping('onednn').set_module_name('e1.0', None).set_module_name('head', None)
    prepared = prepare_fx(copy.deepcopy(net).eval(), qmap, (calib[:1],))
    with torch.no_grad():
        for x in calib.split(4):
            prepared(x)
    return convert_fx(prepared)


def _device(net):
    p = next(iter(net.parameters()), None)
    return p.device if p is not None else torch.device('cpu')


def _run(net, x):
    """Pads the packed input to a multiple of 8 (three poolings) and crops back."""
    h, w = x.shape[-2:]
    xp = F.pad(x, (0, (-w) % 8, 0, (-h) % 8), mode='reflect')
    return net(xp)[..., :2 * h, :2 * w]


def _inputs(mosaic, meta, balance):
    m = cl.normalize(mosaic, meta)
    gain, wb, noise = cl.resolve_meta(m, meta if balance else {**meta, 'wb': (1.0, 1.0, 1.0)})
    raw4 = F.pixel_unshuffle(torch.from_numpy(m)[None, None], 2)
    t = lambda v: torch.tensor([v], dtype=torch.float32)
    x = net_input(raw4, t(gain), torch.tensor([wb], dtype=torch.float32), t(noise[0]), t(noise[1]))
    return x, gain, wb, noise


class HybridISP:
    def __init__(self, net, sharp=0.15):
        self.net, self.sharp = net, sharp

    @torch.no_grad()
    def __call__(self, mosaic, meta):
        x, gain, wb, noise = _inputs(mosaic, meta, balance=True)
        x = x.to(_device(self.net))
        cam = bilinear_demosaic(x[:, :4]) + _run(self.net, x)
        out = render(cam, meta['cam2rgb'])[0].permute(1, 2, 0).cpu().numpy()
        sigma = cl.display_sigma(gain, wb, noise, meta['cam2rgb'])
        return np.clip(cl.sharpen(out, self.sharp / (1 + 30 * sigma)), 0, 1)


class LearnedISP:
    def __init__(self, net):
        self.net = net

    @torch.no_grad()
    def __call__(self, mosaic, meta):
        x, *_ = _inputs(mosaic, meta, balance=False)
        return _run(self.net, x.to(_device(self.net)))[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()


# ---------------------------------------------------------------- real RAW files

def load_raw(path):
    """Reads a camera RAW / DNG with rawpy -> (RGGB mosaic, metadata).

    Black/white levels are applied here (per CFA channel), the CFA is shifted
    to RGGB, and the CCM is built from the camera's XYZ->camera matrix.
    Exposure gain and noise profile are left to be estimated.
    """
    import rawpy
    with rawpy.imread(path) as r:
        mos = r.raw_image_visible.astype(np.float32)
        pat = r.raw_pattern
        black = np.array(r.black_level_per_channel, np.float32)
        idx = np.tile(pat, (mos.shape[0] // 2 + 1, mos.shape[1] // 2 + 1))[:mos.shape[0], :mos.shape[1]]
        mos = (mos - black[idx]) / (r.white_level - black[idx])
        colors = np.array([r.color_desc.decode()[i] for i in pat.ravel()]).reshape(2, 2)
        dy, dx = (int(v) for v in np.argwhere(colors == 'R')[0])
        wb = np.array(r.camera_whitebalance[:3], np.float32)
        xyz2cam = np.array(r.rgb_xyz_matrix[:3], np.float32)
        if xyz2cam.any():
            rgb2cam = xyz2cam @ rs.RGB2XYZ.numpy()
            cam2rgb = np.linalg.inv(rgb2cam / rgb2cam.sum(1, keepdims=True))
        else:  # some DNGs only give LibRaw's ready-made camera->sRGB matrix
            cam2rgb = np.array(r.color_matrix, np.float32)[:, :3]
    mos = mos[dy:, dx:]
    mos = mos[:mos.shape[0] // 2 * 2, :mos.shape[1] // 2 * 2]
    meta = dict(black=0.0, white=1.0, gain=None, noise=None, cam2rgb=cam2rgb,
                wb=tuple(float(v) for v in wb / wb[1]) if wb.all() else None)
    return mos, meta


if __name__ == '__main__':
    from PIL import Image
    ap = argparse.ArgumentParser(description='Process a RAW/DNG file with the hybrid ISP')
    ap.add_argument('raw')
    ap.add_argument('-o', '--out', default='out.png')
    ap.add_argument('--model', default='checkpoints/core.pt')
    ap.add_argument('--classical', action='store_true', help='use the classical pipeline instead')
    args = ap.parse_args()
    mos, meta = load_raw(args.raw)
    out = cl.classical_isp(mos, meta) if args.classical else HybridISP(load_model(args.model))(mos, meta)
    Image.fromarray((out * 255 + 0.5).astype(np.uint8)).save(args.out)
    print('saved', args.out)


In [ ]:
%%writefile train.py
"""Training of the learned ISP components on synthetic RAW from DIV2K.

  core : hybrid ISP core (demosaic + denoise -> camera RGB), from scratch.
         The loss is taken after the classical back end (CCM, gamma, tone),
         so the network is optimised for the final sRGB output.
  e2e  : end-to-end learned ISP (RAW -> sRGB). Adapted / fine-tuned from the
         core weights, so it only has to learn WB, colour and tone on top.
  lite : small core trained by knowledge distillation from the core.

  python train.py core --width 24 --steps 4000
  python train.py e2e  --width 24 --steps 2500 --init checkpoints/core.pt
  python train.py lite --width 12 --steps 3000 --teacher checkpoints/core.pt
"""
import argparse
import glob
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

import raw_sim as rs
from hybrid_isp import bilinear_demosaic, load_model, net_input, render
from model import UNet


def random_crops(imgs, n, p, rng):
    out = []
    for _ in range(n):
        im = imgs[rng.integers(len(imgs))]
        y, x = rng.integers(im.shape[0] - p + 1), rng.integers(im.shape[1] - p + 1)
        c = im[y:y + p, x:x + p]
        if rng.random() < 0.5:
            c = c[:, ::-1]
        if rng.random() < 0.5:
            c = c[::-1]
        if rng.random() < 0.5:
            c = c.transpose(1, 0, 2)
        out.append(np.ascontiguousarray(c))
    return torch.from_numpy(np.stack(out)).permute(0, 3, 1, 2).float() / 255


def synth(srgb, task):
    """Clean sRGB batch -> network input from a random simulated capture."""
    n, dev = srgb.shape[0], srgb.device
    wb, gain, shot, read = (t.to(dev) for t in rs.sample_conditions(n))
    _, raw4 = rs.unprocess(srgb, wb)
    v = lambda t: t.view(-1, 1, 1, 1)
    dn = rs.add_sensor_noise(raw4, v(gain), v(shot), v(read))
    raw4 = (dn - rs.BLACK) / (rs.WHITE - rs.BLACK)
    # the core gets white-balanced input, the end-to-end model has to learn WB itself
    wb_in = torch.ones_like(wb) if task == 'e2e' else wb
    return net_input(raw4, gain, wb_in, shot, read)


def forward(net, x, task):
    if task == 'e2e':
        return net(x)
    return render(bilinear_demosaic(x[:, :4]) + net(x))


@torch.no_grad()
def validate(net, x, y, task):
    out = forward(net.eval(), x, task).clamp(0, 1)
    net.train()
    mse = ((out - y) ** 2).mean((1, 2, 3))
    return float((10 * torch.log10(1 / mse)).mean())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('kind', choices=['core', 'e2e', 'lite'])
    ap.add_argument('--width', type=int, default=24)
    ap.add_argument('--steps', type=int, default=4000)
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--patch', type=int, default=128)
    ap.add_argument('--lr', type=float, default=3e-4)
    ap.add_argument('--init', help='checkpoint to fine-tune from')
    ap.add_argument('--teacher', help='teacher checkpoint (knowledge distillation)')
    ap.add_argument('--name', help='output checkpoint name (default: kind)')
    args = ap.parse_args()
    task = 'e2e' if args.kind == 'e2e' else 'core'
    name = args.name or args.kind
    os.makedirs('checkpoints', exist_ok=True)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.backends.cudnn.benchmark = True

    files = sorted(glob.glob('data/div2k/*.png'))
    imgs = [np.asarray(Image.open(f).convert('RGB')) for f in files]
    train_imgs, val_imgs = imgs[:-5], imgs[-5:]   # last 5 images held out

    torch.manual_seed(123)
    val_y = random_crops(val_imgs, 20, 256, np.random.default_rng(123)).to(dev)
    val_x = synth(val_y, task)
    torch.manual_seed(0)
    rng = np.random.default_rng(0)

    net = UNet(args.width)
    if args.init:
        net.load_state_dict(load_model(args.init).state_dict())
    net = net.to(dev)
    teacher = load_model(args.teacher).to(dev) if args.teacher else None
    opt = torch.optim.Adam(net.parameters(), args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, args.steps, eta_min=args.lr / 100)

    log, t0 = [], time.time()
    print(f'{name}: {sum(p.numel() for p in net.parameters()) / 1e6:.3f}M params on {dev}, '
          f'val psnr at start {validate(net, val_x, val_y, task):.2f}', flush=True)
    for step in range(1, args.steps + 1):
        y = random_crops(train_imgs, args.batch, args.patch, rng).to(dev)
        x = synth(y, task)
        out = forward(net, x, task)
        loss = F.l1_loss(out, y)
        if teacher is not None:
            with torch.no_grad():
                soft = forward(teacher, x, task)
            loss = 0.5 * loss + 0.5 * F.l1_loss(out, soft)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % 250 == 0 or step == args.steps:
            log.append(dict(step=step, loss=round(loss.item(), 5),
                            val_psnr=round(validate(net, val_x, val_y, task), 3),
                            minutes=round((time.time() - t0) / 60, 2)))
            print(json.dumps(log[-1]), flush=True)
            torch.save(dict(width=args.width, kind=task, state=net.state_dict(), log=log),
                       f'checkpoints/{name}.pt')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile prepare_data.py
"""Downloads the datasets used in this project.

Kodak24      -> data/kodak  (test set, 24 images)
DIV2K valid  -> data/div2k  (training set, 100 images, downscaled x2)

The RAW inputs are synthesised from these sRGB images (see raw_sim.py),
so no RAW dataset download is needed.
"""
import io
import os
import zipfile
import urllib.request

from PIL import Image

ROOT = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'data')
KODAK_URL = 'https://r0k.us/graphics/kodak/kodak/kodim{:02d}.png'
DIV2K_URL = 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip'


def get_kodak():
    out = os.path.join(ROOT, 'kodak')
    os.makedirs(out, exist_ok=True)
    for i in range(1, 25):
        path = os.path.join(out, f'kodim{i:02d}.png')
        if not os.path.exists(path):
            urllib.request.urlretrieve(KODAK_URL.format(i), path)
    print('kodak ok')


def get_div2k():
    out = os.path.join(ROOT, 'div2k')
    os.makedirs(out, exist_ok=True)
    if len(os.listdir(out)) >= 100:
        return
    zpath = os.path.join(ROOT, 'div2k.zip')
    if not os.path.exists(zpath):
        print('downloading DIV2K (~450 MB)...')
        urllib.request.urlretrieve(DIV2K_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        for name in z.namelist():
            if not name.endswith('.png'):
                continue
            im = Image.open(io.BytesIO(z.read(name))).convert('RGB')
            # x2 downscale gives cleaner targets with more detail per pixel
            im = im.resize((im.width // 2, im.height // 2), Image.LANCZOS)
            im.save(os.path.join(out, os.path.basename(name)))
    os.remove(zpath)
    print('div2k ok')


if __name__ == '__main__':
    get_kodak()
    get_div2k()


In [ ]:
%%writefile metrics.py
"""Image quality metrics: PSNR, SSIM, colour error (CIEDE2000) and LPIPS."""
import numpy as np
import torch
from skimage.color import deltaE_ciede2000, rgb2lab
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

_lpips = None
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'


def lpips_distance(a, b):
    global _lpips
    if _lpips is None:
        import lpips
        _lpips = lpips.LPIPS(net='alex', verbose=False).eval().to(_dev)
    t = lambda x: (torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1)[None].float() * 2 - 1).to(_dev)
    with torch.no_grad():
        return float(_lpips(t(a), t(b)))


def quality(out, gt):
    """All metrics for one (H,W,3) float image in [0,1] against its ground truth."""
    out = out.astype(np.float32)
    return dict(
        psnr=float(peak_signal_noise_ratio(gt, out, data_range=1)),
        ssim=float(structural_similarity(gt, out, channel_axis=2, data_range=1)),
        delta_e=float(deltaE_ciede2000(rgb2lab(gt), rgb2lab(out)).mean()),
        lpips=lpips_distance(out, gt),
    )


if __name__ == '__main__':
    rng = np.random.default_rng(0)
    gt = rng.random((64, 64, 3)).astype(np.float32)
    same = quality(gt, gt)
    assert same['psnr'] > 99 and same['delta_e'] < 1e-3 and same['lpips'] < 1e-3
    noisy = quality(np.clip(gt + 0.05 * rng.standard_normal(gt.shape), 0, 1).astype(np.float32), gt)
    assert 20 < noisy['psnr'] < 30 and noisy['ssim'] < same['ssim']
    print('ok', noisy)


In [ ]:
%%writefile benchmark.py
"""Efficiency of the learned ISP components on CPU.

Measures parameters, serialized model size, feature-map memory and latency
for a 1 MP frame. Also creates the INT8 models by post-training static
quantisation (calibrated on training crops only) and saves the deployable
INT8 Hybrid-Lite as TorchScript (checkpoints/lite_int8.pt).

    python benchmark.py     -> results/efficiency.{json,md}
"""
import glob
import io
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

import classical_isp as cl
import hybrid_isp as hy
import raw_sim as rs
from train import random_crops, synth

H = W = 1024  # 1 MP output frame


def calibration_inputs(n=64):
    files = sorted(glob.glob('data/div2k/*.png'))[:-5]
    imgs = [np.asarray(Image.open(f).convert('RGB')) for f in files[::2]]
    torch.manual_seed(7)
    return synth(random_crops(imgs, n, 128, np.random.default_rng(7)), 'core')


def size_mb(model):
    buf = io.BytesIO()
    if isinstance(model, torch.jit.ScriptModule):
        torch.jit.save(model, buf)
    else:
        torch.save(model.state_dict(), buf)
    return buf.tell() / 2 ** 20


def feature_memory_mb(model, x):
    """Total size of the feature maps produced by the layers for one frame.

    ReLUs are skipped (in-place in FP32, fused into the conv in INT8).
    """
    total = 0

    def hook(_, __, out):
        nonlocal total
        total += out.numel() * out.element_size()

    layers = [m for m in model.modules() if not list(m.children()) and not isinstance(m, nn.ReLU)]
    hooks = [m.register_forward_hook(hook) for m in layers]
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    return total / 2 ** 20


def latency_ms(fn, *args, runs=10):
    with torch.no_grad():
        for _ in range(2):
            fn(*args)
        times = []
        for _ in range(runs):
            t = time.perf_counter()
            fn(*args)
            times.append(time.perf_counter() - t)
    return 1000 * float(np.median(times))


def export_int8(calib=None):
    """Quantises Hybrid-Lite to INT8 and saves it as TorchScript (deployable model)."""
    torch.backends.quantized.engine = 'onednn'
    calib = calibration_inputs() if calib is None else calib
    lite_q = hy.quantize_int8(hy.load_model('checkpoints/lite.pt'), calib)
    torch.jit.save(torch.jit.trace(lite_q, calib[:1]), 'checkpoints/lite_int8.pt')
    return lite_q


def main():
    os.makedirs('results', exist_ok=True)
    calib = calibration_inputs()
    lite_q = export_int8(calib)
    core, e2e, lite = (hy.load_model(f'checkpoints/{n}.pt') for n in ('core', 'e2e', 'lite'))
    core_q = hy.quantize_int8(core, calib)

    x = torch.rand(1, 8, H // 2, W // 2)
    rows = []
    for name, net, fp32 in [('Learned e2e (FP32)', e2e, e2e), ('Hybrid core (FP32)', core, core),
                            ('Hybrid core (INT8)', core_q, core), ('Hybrid-Lite, distilled (FP32)', lite, lite),
                            ('Hybrid-Lite, distilled (INT8)', lite_q, lite)]:
        rows.append(dict(model=name, params_M=sum(p.numel() for p in fp32.parameters()) / 1e6,
                         size_MB=size_mb(net), feature_mem_MB=feature_memory_mb(net, x),
                         latency_ms=latency_ms(net, x)))
        print(rows[-1], flush=True)

    # complete pipelines on a 1 MP RAW frame (includes the classical stages)
    gt = np.asarray(Image.open('data/kodak/kodim01.png').convert('RGB').resize((W, H)), np.float32) / 255
    mosaic, meta = rs.capture(gt, **rs.CONDITIONS['low_light'])
    deploy = torch.jit.load('checkpoints/lite_int8.pt')
    pipes = [('Classical ISP', cl.classical_isp), ('Learned ISP (e2e)', hy.LearnedISP(e2e)),
             ('Hybrid ISP', hy.HybridISP(core)), ('Hybrid-Lite ISP (FP32)', hy.HybridISP(lite)),
             ('Hybrid-Lite ISP (INT8)', hy.HybridISP(deploy))]
    full = [dict(pipeline=n, latency_ms=latency_ms(p, mosaic, meta, runs=5)) for n, p in pipes]
    print(full, flush=True)

    res = dict(threads=torch.get_num_threads(), frame=f'{W}x{H}', models=rows, pipelines=full)
    with open('results/efficiency.json', 'w') as f:
        json.dump(res, f, indent=1)
    lines = [f'Measured on CPU ({torch.get_num_threads()} threads), {W}x{H} output frame.', '',
             '| Model | Params (M) | Size (MB) | Feature maps / frame (MB) | Latency (ms) |',
             '|---|---:|---:|---:|---:|']
    lines += [f"| {r['model']} | {r['params_M']:.3f} | {r['size_MB']:.2f} | {r['feature_mem_MB']:.0f} | "
              f"{r['latency_ms']:.0f} |" for r in rows]
    lines += ['', '| Full pipeline (RAW in, sRGB out) | Latency (ms) |', '|---|---:|']
    lines += [f"| {p['pipeline']} | {p['latency_ms']:.0f} |" for p in full]
    with open('results/efficiency.md', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile evaluate.py
"""Quality comparison of classical, learned (end-to-end) and hybrid ISP.

RAW captures of the Kodak24 images are simulated under several lighting and
colour scenarios. Every pipeline gets exactly the same RAW + metadata.

    python evaluate.py      -> results/metrics.{json,md}, results/*.png
"""
import glob
import json
import os

import matplotlib
import numpy as np
import torch
from PIL import Image

import classical_isp as cl
import hybrid_isp as hy
import raw_sim as rs
from metrics import quality

matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402

SCENARIOS = {
    **{k: dict(v) for k, v in rs.CONDITIONS.items()},
    # warm, tungsten-like illuminant outside the training range of the WB gains
    'unseen_illuminant': dict(gain=4, shot=5e-4, wb=(1.3, 1.0, 2.6)),
    # no as-shot WB and no noise profile: AWB + blind noise estimation
    'no_metadata': dict(gain=4, shot=5e-4, blind=True),
}
# (image, scenario) -> crop (y, x, size) for the visual comparisons
FIGURES = {('kodim19', 'low_light'): (60, 180, 200),
           ('kodim23', 'night'): (60, 260, 240),
           ('kodim08', 'indoor'): (40, 300, 200),
           ('kodim03', 'unseen_illuminant'): (120, 200, 260)}
ABLATION = 'Hybrid (fixed noise map)'


def pipelines():
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'  # INT8 model always runs on CPU
    core, e2e, lite = (hy.load_model(f'checkpoints/{n}.pt').to(dev) for n in ('core', 'e2e', 'lite'))
    torch.backends.quantized.engine = 'onednn'
    hybrid = hy.HybridISP(core)
    return {
        'Classical': cl.classical_isp,
        'Learned (e2e)': hy.LearnedISP(e2e),
        'Hybrid': hybrid,
        'Hybrid-Lite': hy.HybridISP(lite),
        'Hybrid-Lite INT8': hy.HybridISP(torch.jit.load('checkpoints/lite_int8.pt')),
        # ablation: the core always gets the noise map of the 'indoor' level,
        # i.e. no adaptation to the real noise of the frame
        ABLATION: lambda mos, meta: hybrid(mos, {**meta, 'noise': (
            5e-4 * 4 / meta['gain'], rs.read_noise(5e-4) * (4 / meta['gain']) ** 2)}),
    }


def write_tables(summary):
    scen = list(summary)
    methods = list(summary[scen[0]])
    lines = []
    for key, label, fmt, best in [('psnr', 'PSNR (dB) &uarr;', '{:.2f}', max),
                                  ('ssim', 'SSIM &uarr;', '{:.3f}', max),
                                  ('delta_e', 'Colour error &Delta;E2000 &darr;', '{:.2f}', min),
                                  ('lpips', 'LPIPS &darr;', '{:.3f}', min)]:
        lines += [f'#### {label}', '', '| Method | ' + ' | '.join(scen) + ' |',
                  '|---|' + '---:|' * len(scen)]
        top = {s: best(v[key] for m, v in summary[s].items() if m != ABLATION) for s in scen}
        for m in methods:
            cells = []
            for s in scen:
                if m not in summary[s]:
                    cells.append('-')
                    continue
                v = fmt.format(summary[s][m][key])
                cells.append(f'**{v}**' if summary[s][m][key] == top[s] and m != ABLATION else v)
            lines.append(f'| {m} | ' + ' | '.join(cells) + ' |')
        lines.append('')
    with open('results/metrics.md', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


def plot_robustness(summary):
    conds = [c for c in rs.CONDITIONS if c in summary]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for m in summary[conds[0]]:
        for ax, key in zip(axes, ('psnr', 'delta_e', 'lpips')):
            ax.plot(conds, [summary[c][m][key] for c in conds], 'o-', label=m,
                    ls='--' if m == ABLATION else '-')
    for ax, title in zip(axes, ('PSNR (dB), higher is better', 'Delta E 2000, lower is better',
                                'LPIPS, lower is better')):
        ax.set_title(title)
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels([f"{c}\n(x{rs.CONDITIONS[c]['gain']})" for c in conds])
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    fig.suptitle('Robustness to illumination / noise (digital gain in brackets; x64 is outside the training range)')
    fig.tight_layout()
    fig.savefig('results/robustness.png', dpi=120)
    plt.close(fig)


def plot_examples(keep):
    for (name, scen), (y, x, s) in FIGURES.items():
        items = [(m, *keep[(name, scen, m)]) for m in ['Ground truth', 'Classical', 'Learned (e2e)', 'Hybrid',
                                                       'Hybrid-Lite INT8'] if (name, scen, m) in keep]
        fig, axes = plt.subplots(1, len(items), figsize=(3 * len(items), 3.4))
        for ax, (m, img, q) in zip(axes, items):
            ax.imshow(np.clip(img[y:y + s, x:x + s], 0, 1))
            ax.set_title(m if q is None else f"{m}\n{q['psnr']:.2f} dB, dE {q['delta_e']:.2f}", fontsize=9)
            ax.axis('off')
        fig.suptitle(f'{name} - {scen}')
        fig.tight_layout(rect=(0, 0, 1, 0.9))
        fig.savefig(f'results/example_{name}_{scen}.png', dpi=110)
        plt.close(fig)


def plot_training():
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for n, label in [('core', 'Hybrid core (from scratch)'), ('e2e', 'Learned e2e (fine-tuned from core)'),
                     ('lite', 'Hybrid-Lite (distilled from core)')]:
        log = torch.load(f'checkpoints/{n}.pt', weights_only=False)['log']
        ax.plot([r['step'] for r in log], [r['val_psnr'] for r in log], label=label)
    ax.set_xlabel('training step')
    ax.set_ylabel('validation PSNR (dB)')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig('results/training.png', dpi=120)
    plt.close(fig)


def real_raw_demo(methods, path='data/raw/pixel4a.dng', side=768):
    """Real smartphone RAW (no ground truth): centre crop of a Pixel 4a DNG."""
    if not os.path.exists(path):
        return
    mos, meta = hy.load_raw(path)
    y, x = (mos.shape[0] - side) // 4 * 2, (mos.shape[1] - side) // 4 * 2
    mos = mos[y:y + side, x:x + side]
    names = ['Classical', 'Learned (e2e)', 'Hybrid', 'Hybrid-Lite INT8']
    fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4.4))
    for ax, n in zip(axes, names):
        ax.imshow(methods[n](mos, meta))
        ax.set_title(n)
        ax.axis('off')
    fig.suptitle('Real RAW input: Google Pixel 4a DNG (raw.pixls.us, CC0), centre crop')
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig('results/real_raw_pixel4a.png', dpi=110)
    plt.close(fig)


def main():
    os.makedirs('results', exist_ok=True)
    files = sorted(glob.glob('data/kodak/*.png'))
    methods = pipelines()
    scores = {s: {m: [] for m in methods} for s in SCENARIOS}
    noise_ratio = {s: [] for s in rs.CONDITIONS}  # blind estimate / true noise std at mid grey
    keep = {}
    for k, (scen, cfg) in enumerate(SCENARIOS.items()):
        for i, f in enumerate(files):
            name = os.path.basename(f)[:-4]
            gt = np.asarray(Image.open(f).convert('RGB'), np.float32) / 255
            rng = np.random.default_rng(i)  # same illuminant per image in every scenario
            wb = cfg.get('wb') or (float(rng.uniform(1.6, 2.6)), 1.0, float(rng.uniform(1.3, 2.1)))
            mosaic, meta = rs.capture(gt, cfg['gain'], cfg['shot'], wb, seed=1000 * k + i)
            if cfg.get('blind'):
                meta = {**meta, 'wb': None, 'noise': None}
            if scen in noise_ratio:
                (s0, r0), (s1, r1) = cl.estimate_noise(cl.normalize(mosaic, meta)), meta['noise']
                noise_ratio[scen].append(float(np.sqrt((s0 * 0.2 + r0) / (s1 * 0.2 + r1))))
            for m, isp in methods.items():
                if m == ABLATION and scen not in rs.CONDITIONS:
                    continue
                out = isp(mosaic, meta)
                q = quality(out, gt)
                scores[scen][m].append(q)
                if (name, scen) in FIGURES:
                    keep[(name, scen, m)] = (out, q)
            if (name, scen) in FIGURES:
                keep[(name, scen, 'Ground truth')] = (gt, None)
        print(scen, {m: round(np.mean([q['psnr'] for q in v]), 2) for m, v in scores[scen].items() if v},
              flush=True)

    summary = {s: {m: {key: float(np.mean([q[key] for q in v])) for key in v[0]}
                   for m, v in d.items() if v} for s, d in scores.items()}
    with open('results/metrics.json', 'w') as f:
        json.dump(summary, f, indent=1)
    write_tables(summary)
    with open('results/metrics.md', 'a', encoding='utf-8') as f:
        f.write('\n#### Blind noise estimation (estimated / true noise std at mid grey)\n\n'
                '| Scenario | median | min | max |\n|---|---:|---:|---:|\n')
        f.writelines(f'| {s} | {np.median(v):.2f} | {min(v):.2f} | {max(v):.2f} |\n'
                     for s, v in noise_ratio.items() if v)
    plot_robustness(summary)
    plot_examples(keep)
    plot_training()
    real_raw_demo(methods)


if __name__ == '__main__':
    main()


In [ ]:
!pip -q install lpips rawpy
!python prepare_data.py
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'
!mkdir -p data/raw && wget -q -O data/raw/pixel4a.dng "https://raw.pixls.us/download/data/Google/Pixel%204a/PXL_20201121_100251397.dng" && ls -la data/raw

In [ ]:
!python train.py core --width 24 --steps 15000 --batch 32
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'

In [ ]:
!python train.py e2e --width 24 --steps 8000 --batch 32 --init checkpoints/core.pt
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'

In [ ]:
!python train.py lite --width 16 --steps 12000 --batch 32 --teacher checkpoints/core.pt
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'

In [ ]:
!python -c "import benchmark; benchmark.export_int8()"
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'

In [ ]:
!python evaluate.py
assert globals().get('_exit_code', 0) == 0, 'step failed, see output above'
!cat results/metrics.md

In [ ]:
!zip -qr hybrid_isp_colab_results.zip checkpoints results && ls -la hybrid_isp_colab_results.zip
from google.colab import files
files.download('hybrid_isp_colab_results.zip')